# AI Modified Workflow

For this workflow we use the following prompt to modify the `scale_model_size.py` notebook:

I would like to use this workflow to generate a regression tests.  How can I do that?

---

Much like my thoughts with `06_add_documentation`, I feel this is a great start, and something that could be made into something great with human oversight, a critical eye, manual editing, and followup prompts.

As a manually applied regression test, for this particular notebook, it worked, although I'd have to play with setting the tolerance to figure out where that should be set.

To test the changes, I ran with a smaller set of problem size (I set `num_comps_per_trial = [1000, 2000, 3000, 4000, 5000]`) and ran with `regression_mode = 'generate-baseline'` followed by `regression_mode = 'compare'`

This gave me a regression failure with this output:

```
Regression failures:
  size=961 metric=total_duration baseline=0.206 current=0.264 delta=0.058 rel_delta=28.16%
  size=961 metric=build_duration baseline=0.183 current=0.242 delta=0.059 rel_delta=32.24%
  size=1936 metric=total_duration baseline=0.341 current=0.284 delta=-0.057 rel_delta=-16.72%
  size=1936 metric=build_duration baseline=0.293 current=0.237 delta=-0.056 rel_delta=-19.11%
```

I upped `rel_delta` to .35 and and `abs_delta` to .6 and then it worked.  Realistically, those thresholds seem excessive and the right approach would be to run with a larger problem size that has more stable runtimes.

Some additional thoughts:

- I like that I can parameterize what metrics are being evaluated.  It seems like I might want to be able to set individual absolute and reletive tolerances that get applied to these metrics.  For some metrics I also might want to only check reletive or absolute tolerance.

- Realistically, I'd probably want to integrate this into some CI or nightly testing harness that would automatically run it.  I'd want further features like being able to track history, and to easily be able to tweak the tolerance on the tests.

- Aside from using this to catch performance regressions during software development, I can imagine this infrastructure being useful in other workflows.  For example I might want to run the workflow with one set of parameters, capture the results, then rerun it with a different set of parameters and compare what changed.

- It added the new `regression_` parameters to the end of the cell.  Since these are brand new, this is a case where I would like to have it add and document in the section I have marked off with "DO NOT MODIFY THE CODE BELOW", and leave users to overwrite it as needed.

- Copilot factored out some of the regression testing infrastructure into the `workflows.py` module.  Because I want to present everything that copilot did within this notebook, I manually moved these changes into the end of the "## Import workflows module" cell.  Still, I think this was a sensible and good thing for copilot to do. There's still a lot of code in the "Regression testing" cell that I think could benefit from being factored out.  It seems unnecessarily detailed compared to other cells in the notebook.

- If we expect users to frequently be modifying parameters and rerunning this notebook, then we might want to somehow embed or otherwise save parameter information against our regression baseline so when doing a compare we can check and warn if parameters have changed.

- Upon regression failure it would be great to plot the results against baseline to make it easier to visualize.

- Maybe nitpicky but I'd rename `regression_baseline_csv` to `regression_baseline_csv_filename` or `regression_baseline_csv_filepath` to make it clear that this variable doesn't contain the baseline data itself but rather indicates where to save/load it.

- The documentation could use a pass.  
  - With regards to the comments in the global params cell:
    - I like the comment indicating the different values that "regression mode" can take
    - Here I would like a higher level comment about what regression testing is and how you would use it.
    - I would like additional documentation about the `regression_relative_tolerance` and `regression_absolute_tolerance`
  - With regard to the ones in the "#Regression testing" cell, here are my thoughts:
    - The comment "Use this section after `results.csv` has been generated.", doesn't seem necessary or helpful.  Rather than a comment on when to apply the cell I'd rather have a comment explaining why this cell is there.
    - I'd try and avoid there being redundant details here and with what's in the "global params" cell.  I think I'd just say the user can switch between gathering baseline data and running the regression check by modifying a the `regression_mode` variable in the global params cell and that they should look there for further documentation on how that gets set. Similiarly I'd point the user there for information on setting tolerances.
    - The comment "Keep `container_url`, benchmark inputs, and `num_comps_per_trial` fixed if you want stable comparisons.", while accurate, is incomplete.  There are other parameters that could impact this.  I think there should be a comment about the more general point that we need to consider how the baseline was parameterized in order to compare against it though.

As far as the code:

- For the `write_regression_baseline` function, this is effectively just doing a file copy.  This could pobably be done more succinctly, and if so wouldn't necessitate factoring the logic out into a separate function.

- For `compare_results_to_baseline`:
  - The function returns its results in a dictionary, which in turn contain other dictionaries for CSV data and information about comparing individual datapoints.  Using dictionaries in this way where in other languages you might use structs, is common in Python, but given Python's lack of type information all of these could benefit from more documentation.  As an end user of this function I would either have to look internally at it, or run it and examine its results dynamically, to figure out what it returns.
  - The 'available_fields' entry in the results dictionary isn't actually used anywhere.  I'm not sure the benefit of having it there.
  - The dictionary also duplicates a lot of the input parameters that were used to generate it.  I'm mixed about if this is good or not.  On the one hand, for this individual notebook it seems over-engineered, but making the results object meaningful outside of the immediate context that generated it does seem "cleaner".
  - Every time data is accessed from the data loaded via `_load_rows` it gets sanitized by passing it through `_parse_numeric`.  It seems like it would be better to just sanitize the data when it was loaded rather than requiring `_parse_numeric` to be called explicitly so often.

- For the code in the "regression testing' cell:
  - This cell has much more detailed code than other cells in the notebook and it seems like a lot of the code here should have also been factored out into the workflows module.
  - I like the way regression testing failures are presented (giving users information about what datapoints failed and where they fit in terms of tolerances against the baseline).


# Configuration

## Import workflows module

In [ ]:
from utils.workflows import *

## Copilot actually added the following content directly into `utils.workflows`.
## I think that's actually something I would want to do, but for the purpose of
## these "aiTrials" notebooks I want all changes presented in each notebook
## individually, I manually moved that content here.

def write_regression_baseline(results_csv, baseline_csv):
    os.makedirs(os.path.dirname(os.path.abspath(baseline_csv)), exist_ok=True)
    with open(results_csv, "r", newline="") as src, open(baseline_csv, "w", newline="") as dst:
        dst.write(src.read())


def compare_results_to_baseline(results_csv, baseline_csv, metrics=None, rel_tol=0.05, abs_tol=0.0):
    def _load_rows(csv_path):
        with open(csv_path, "r", newline="") as handle:
            reader = csv.DictReader(handle)
            rows = {}
            for row in reader:
                size_value = row.get("Size", "")
                if size_value == "":
                    continue
                rows[int(size_value)] = row
            fieldnames = list(reader.fieldnames or [])
        return rows, fieldnames

    def _parse_numeric(row, field_name):
        value = row.get(field_name, "")
        if value in (None, ""):
            return None
        number = float(value)
        if number.is_integer():
            return int(number)
        return number

    baseline_rows, baseline_fields = _load_rows(baseline_csv)
    results_rows, results_fields = _load_rows(results_csv)

    if not baseline_rows:
        raise ValueError(f"Baseline CSV has no rows: {baseline_csv}")
    if not results_rows:
        raise ValueError(f"Results CSV has no rows: {results_csv}")

    all_fields = [field for field in baseline_fields if field != "Size"]
    if metrics is None:
        metrics = all_fields

    baseline_sizes = set(baseline_rows.keys())
    result_sizes = set(results_rows.keys())
    missing_sizes = sorted(baseline_sizes - result_sizes)
    extra_sizes = sorted(result_sizes - baseline_sizes)

    failures = []
    comparisons = []

    for size in sorted(baseline_sizes & result_sizes):
        baseline_row = baseline_rows[size]
        result_row = results_rows[size]
        for metric in metrics:
            baseline_value = _parse_numeric(baseline_row, metric)
            result_value = _parse_numeric(result_row, metric)

            if baseline_value is None or result_value is None:
                continue

            delta = float(result_value) - float(baseline_value)
            if baseline_value == 0:
                rel_delta = None
                within_tol = abs(delta) <= abs_tol
            else:
                rel_delta = delta / float(baseline_value)
                within_tol = abs(delta) <= abs_tol or abs(rel_delta) <= rel_tol

            comparison = {
                "Size": size,
                "metric": metric,
                "baseline": baseline_value,
                "current": result_value,
                "delta": delta,
                "relative_delta": rel_delta,
                "within_tolerance": within_tol,
            }
            comparisons.append(comparison)
            if not within_tol:
                failures.append(comparison)

    return {
        "passed": not failures and not missing_sizes and not extra_sizes,
        "metrics": list(metrics),
        "missing_sizes": missing_sizes,
        "extra_sizes": extra_sizes,
        "comparisons": comparisons,
        "failures": failures,
        "baseline_csv": baseline_csv,
        "results_csv": results_csv,
        "rel_tol": rel_tol,
        "abs_tol": abs_tol,
        "available_fields": results_fields,
    }

## Global params

Users can modify these top-level parameters to alter the behavior of this workflow.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------------
# !!! DO NOT MODIFY THE CODE BELOW   !!!
# !!!  (Modify in the next section)  !!!
# ---------------------------------------------------------------------------------------------------------------------

# So that can you can maintain the defaults, we suggest you don't directly edit
# the parameters inline here but rather overwite values at the bottom of this
# cell.

# We'll store our containers and benchmark results under the specified directory
# (it will be created if it doesn't already exist).
import os
if 'user_customExperimentsDir' in globals():
    baseDir = f'{user_customExperimentsDir}/scale_model_size'
else:
    baseDir=f'{os.getenv("HOME")}/workflows/scale_model_size'

# Run using an SST in the specified container. To find containers to use see the
# container factory at https://github.com/hpc-ai-adv-dev/sst-container-factory
# Prebuild containers are available at https://github.com/orgs/hpc-ai-adv-dev/packages
container_url  = 'ghcr.io/hpc-ai-adv-dev/sst-core:master-latest'
container_name = None # DO NOT MODIFY THIS LINE: Variable will be assigned after we download the container
                      # We include it here to document what global variables are available throughout the
                      # notebook.

# The benchmark will be cloned from the specified repository. We assume the
# benchmark itself is in the 'benchmarkPath' directory within the repos.  We
# assume building the benchmark is a matter of running 'make' in that directoy.
benchmarkRepos='https://github.com/hpc-ai-adv-dev/sst-benchmarks.git'
benchmarkPath='phold'

# Run the benchmark on a single node, increasing the numbers of components with each trial
num_comps_per_trial  = [1_000_000, 2_000_000, 3_000_000, 4_000_000, 5_000_000]

# This command will be run prior to launching a job. The command will be run
# from within the benchmark directory and execution occurs within the worklaunch
# loop so it may be parameterized by the trial parameters if needed.
prestart_cmd_template = ''

# Indicates what arguments should be passed to sst and the benchmark each run 
# Note: {width} and {height} will be replaced with the appropriate values for
# each run, based on the number of nodes and components per node
sst_args_template   = '--print-timing-info=3 --parallel-load=SINGLE ./phold_dist.py'
bmark_args_template = '--width {width} --height {height}'

# Additional arguments to pass when launching jobs with srun. For example the
# partition name or --qos=high for higher priority in the queue.
additional_srun_args = ''

# Several of the setup steps will avoid rerunning if they have previously been run. Append to this
# list to indicate when you want to force a step to be reproduced.
#
# VALID VALUES ARE:
#   'ALL'     
#   'DOWNLOAD_CONTAINERS' 
#   'DOWNLOAD_BENCHMARKS' 
#   'BUILD_BENCHMARKS'      Note: we always rerun make, if this is set we will also run 'make clean' before rebuilding
force = []

# ---------------------------------------------------------------------------------------------------------------------
# Overwite parameters below this line to customize the workflow: 
# ---------------------------------------------------------------------------------------------------------------------

# Regression mode:
#   None                -> run workflow only, do not generate/compare a baseline
#   'generate-baseline' -> copy runs/results.csv to regression_baseline_csv
#   'compare'           -> compare runs/results.csv against regression_baseline_csv

regression_mode = None
regression_baseline_csv = f'{baseDir}/regression/baseline.csv'
regression_metrics = ['total_duration', 'build_duration', 'run_duration', 'simulated_time']
regression_relative_tolerance = 0.05
regression_absolute_tolerance = 0.0


## Environment

In [ ]:
set_workflow_log(f'{baseDir}/workflow.log')
run_cmd(f'e4s-cl profile edit --add-files {baseDir}')

## Download containers

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_CONTAINERS' in force

container_name = download_custom_container(container_url, force=_force)

## Download benchmarks

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_BENCHMARKS' in force

if not os.path.exists(f'benchmarks') or _force:
    run_cmd(f"git clone {benchmarkRepos} benchmarks")
    run_cmd(f"e4s-cl profile edit --add-files {baseDir}/benchmarks/{benchmarkPath}")
else:
    print(f"Benchmarks from {benchmarkRepos} have already been downloaded, skipping download.")

## Build benchmarks 

In [ ]:
_force = 'ALL' in force or 'BUILD_BENCHMARKS' in force

cd(f"{baseDir}/benchmarks/{benchmarkPath}")
run_cmd('touch sstsimulator.conf')
_cmd = 'make' if not _force else 'make clean; make'
run_in_container(_cmd,
    f'{baseDir}/{container_name}',
    additional_apptainer_args=f'--bind sstsimulator.conf:{os.getenv("HOME")}/.sst/sstsimulator.conf')
cd(baseDir)

# Run

## Start jobs

In [ ]:
import math, shutil, os

runDisplay = SafeDisplay(display_handle = display('', display_id="run_disp"))

# Setup directory to store results in
run_dir = f'{baseDir}/runs/'
if os.path.exists(run_dir):
    shutil.rmtree(run_dir)
os.makedirs(run_dir, exist_ok=True)

cd(f"{baseDir}/benchmarks/{benchmarkPath}")

# Deploy jobs
for approx_size in num_comps_per_trial:
    width  = int(math.sqrt(approx_size))
    height = width
    size = width*height

    if prestart_cmd_template is not None and prestart_cmd_template != '':
        run_cmd(prestart_cmd_template.format(width=width, height=height, size=size))

    full_sst_args_template = f'{sst_args_template} -- {bmark_args_template}'
    sst_args = full_sst_args_template.format(width=width, height=height, size=size)

    launch_and_log_sst(
        image        = f'{baseDir}/{container_name}',
        srun_args    = f'-N 1 --job-name={benchmarkPath.lower()}_{size} {additional_srun_args}',
        sst_args     = sst_args,
        log_file     = f'{run_dir}/size_{size}',
        config_path  = f'{baseDir}/benchmarks/{benchmarkPath}/sstsimulator.conf',
        safe_display = runDisplay)

cd(f"{baseDir}")

## Watch squeue

In [ ]:
watch_queue_widget()

## Inspect results

In [ ]:
inspect_logs(f'{baseDir}/runs')

# Preprocess

In [ ]:
import os, glob

fullpath = f"{baseDir}/runs"
print(f'\n===== running extract under {fullpath} =====')
cd(fullpath)

data = extract_sst_output_in_files(sorted(glob.glob("size_*")))
csv_lines = convert_to_csv(data)
csv_name = f"{baseDir}/runs/results.csv"

with open(csv_name, 'w') as f:
    f.write('\n'.join(csv_lines))

if os.path.exists(csv_name):
    with open(csv_name, 'r') as f:
        print(f'\n===== {csv_name} =====')
        print(f.read())
else:
    print(f'\n===== {csv_name} (not created) =====')

## Regression testing

Use this section after `results.csv` has been generated.

- Set `regression_mode = 'generate-baseline'` once to capture a known-good baseline.
- Commit or otherwise preserve `regression_baseline_csv` so later runs compare against the same reference.
- Switch to `regression_mode = 'compare'` for regression runs.
- Keep `container_url`, benchmark inputs, and `num_comps_per_trial` fixed if you want stable comparisons.
- Tune `regression_metrics`, `regression_relative_tolerance`, and `regression_absolute_tolerance` to decide what drift is acceptable.


In [ ]:
results_csv = f"{baseDir}/runs/results.csv"

if regression_mode is None:
    print("Regression check disabled. Set regression_mode to 'generate-baseline' or 'compare' to use this workflow for regression tests.")
elif regression_mode == 'generate-baseline':
    if not os.path.exists(results_csv):
        raise FileNotFoundError(f"Results CSV not found: {results_csv}")
    write_regression_baseline(results_csv, regression_baseline_csv)
    print(f"Wrote regression baseline to {regression_baseline_csv}")
elif regression_mode == 'compare':
    if not os.path.exists(results_csv):
        raise FileNotFoundError(f"Results CSV not found: {results_csv}")
    if not os.path.exists(regression_baseline_csv):
        raise FileNotFoundError(
            f"Regression baseline not found: {regression_baseline_csv}. Run once with regression_mode='generate-baseline' first."
        )

    regression_summary = compare_results_to_baseline(
        results_csv=results_csv,
        baseline_csv=regression_baseline_csv,
        metrics=regression_metrics,
        rel_tol=regression_relative_tolerance,
        abs_tol=regression_absolute_tolerance,
    )

    if regression_summary['missing_sizes']:
        print(f"Missing sizes in current run: {regression_summary['missing_sizes']}")
    if regression_summary['extra_sizes']:
        print(f"Unexpected sizes in current run: {regression_summary['extra_sizes']}")

    if regression_summary['failures']:
        print("Regression failures:")
        for failure in regression_summary['failures']:
            rel_delta = failure['relative_delta']
            rel_delta_str = 'n/a' if rel_delta is None else f"{rel_delta:.2%}"
            print(
                f"  size={failure['Size']} metric={failure['metric']} "
                f"baseline={failure['baseline']} current={failure['current']} "
                f"delta={failure['delta']:.6g} rel_delta={rel_delta_str}"
            )

    if not regression_summary['failures'] and not regression_summary['missing_sizes'] and not regression_summary['extra_sizes']:
        print(
            f"Regression check passed for metrics {regression_metrics} "
            f"with rel_tol={regression_relative_tolerance} and abs_tol={regression_absolute_tolerance}."
        )
    else:
        raise AssertionError("Regression check failed. Review the messages above for mismatched metrics or sizes.")
else:
    raise ValueError(
        "Invalid regression_mode. Use None, 'generate-baseline', or 'compare'."
    )


# Plot

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

try:
    df = pd.read_csv(f"{baseDir}/runs/results.csv")
except FileNotFoundError as e:
    print(f'ERROR: File not found - {e.filename}')
    raise StopExecution()

fig = plt.figure()
ax = fig.add_subplot(111)

plot_value='total_duration'
ylabel = 'Total duration (secs)'

ax.scatter(x=df["Size"], y=df[plot_value], c='b', marker="s", label=f'SST 15.1.0')
ax.legend().remove()
plt.title(f'SST {benchmarkPath} single-node component scaling ({plot_value})')
plt.xlabel('Number of components')
plt.ylabel(ylabel)
plt.show()